In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

In [1]:
import pandas as pd

df = pd.read_csv("tife_dataset_full.csv")
print(df.shape)
df.head()

(159, 17)


,NO,Ti,Fe,Zr,V,Cr,Mn,Ni,Co,Y,T,Size1,Size2,Pa,Pd,wt,ref
0,1,1.0,0.85,0.00,0.0,0.15,0.0,0.0,0.0,0.0,25,45,50,0.060,0.040,1.38,[1]
1,2,1.0,0.80,0.05,0.0,0.15,0.0,0.0,0.0,0.0,25,45,50,0.032,0.020,1.42,[1]
2,3,1.0,0.75,0.10,0.0,0.15,0.0,0.0,0.0,0.0,25,45,50,0.027,0.020,1.17,[1]
3,4,1.0,0.70,0.15,0.0,0.15,0.0,0.0,0.0,0.0,25,45,50,0.018,0.018,0.90,[1]
4,5,1.0,0.90,0.00,0.0,0.00,0.0,0.1,0.0,0.0,50,270,830,0.200,0.150,1.70,[2]


In [2]:
print(df.shape)
print(df.info()) # data types, non-nan values

(159, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   NO      159 non-null    int64  
 1   Ti      159 non-null    float64
 2   Fe      159 non-null    float64
 3   Zr      159 non-null    float64
 4   V       159 non-null    float64
 5   Cr      159 non-null    float64
 6   Mn      159 non-null    float64
 7   Ni      159 non-null    float64
 8   Co      159 non-null    float64
 9   Y       159 non-null    float64
 10  T       159 non-null    int64  
 11  Size1   159 non-null    int64  
 12  Size2   159 non-null    int64  
 13  Pa      159 non-null    float64
 14  Pd      159 non-null    float64
 15  wt      159 non-null    float64
 16  ref     159 non-null    object 
dtypes: float64(12), int64(4), object(1)
memory usage: 21.2+ KB
None


In [4]:
# check missing values
print("\nMissing values per column:")
print(df.isna().sum())


# check if any row has missing values
print("\nRows with missing values:")
print(df[df.isna().any(axis=1)]) # problematic rows


Missing values per column:
NO       0
Ti       0
Fe       0
Zr       0
V        0
Cr       0
Mn       0
Ni       0
Co       0
Y        0
T        0
Size1    0
Size2    0
Pa       0
Pd       0
wt       0
ref      0
dtype: int64

Rows with missing values:
Empty DataFrame
Columns: [NO, Ti, Fe, Zr, V, Cr, Mn, Ni, Co, Y, T, Size1, Size2, Pa, Pd, wt, ref]
Index: []


In [6]:
# features: alloy composition, temperature, particle size
feature_cols = ["Ti", "Fe", "Zr", "V", "Cr", "Mn", "Ni", "Co", "Y", "T", "Size1", "Size2"]

# targets: absorption pressure, desorption pressure, hydrogen capacity
target_cols = ["Pa", "Pd", "wt"]

# create X and y
X = df[feature_cols]
y_pa = df["Pa"]
y_pd = df["Pd"]
y_wt = df["wt"]

print("X shape:", X.shape)
print("Targets:", target_cols)

X shape: (159, 12)
Targets: ['Pa', 'Pd', 'wt']


In [11]:
X = df[feature_cols]
y = df["Pa"] # predict Pa


X_train, X_test, y_train, y_test = train_test_split(# split data
    X, y, test_size=0.2, random_state=42
)

In [15]:
for target in ["Pa", "Pd", "wt"]:
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    rf.fit(X_train, y_train)
    xgb.fit(X_train, y_train)

    y_rf = rf.predict(X_test)
    y_xgb = xgb.predict(X_test)

    print(f"\nTarget: {target}")
    print("Random Forest:")
    print("R2:", r2_score(y_test, y_rf))
    print("MAE:", mean_absolute_error(y_test, y_rf))
    print("RMSE:", root_mean_squared_error(y_test, y_rf))

    print("\nXGBoost:")
    print("R2:", r2_score(y_test, y_xgb))
    print("MAE:", mean_absolute_error(y_test, y_xgb))
    print("RMSE:", root_mean_squared_error(y_test, y_xgb))



Target: Pa
Random Forest:
R2: 0.5920362101586594
MAE: 0.18776604687499973
RMSE: 0.3500937190352234

XGBoost:
R2: 0.8534086763140037
MAE: 0.12497923813285305
RMSE: 0.20985902155108296

Target: Pd
Random Forest:
R2: 0.7795345128917435
MAE: 0.09029165624999998
RMSE: 0.17943834449844817

XGBoost:
R2: 0.9149887786868124
MAE: 0.0657639752669027
RMSE: 0.11142511100426714

Target: wt
Random Forest:
R2: 0.5520673030245331
MAE: 0.16082203124999986
RMSE: 0.23704161989676245

XGBoost:
R2: 0.6694086379102003
MAE: 0.13774769624322653
RMSE: 0.2036404156322526


In [18]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "r2": "r2",
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error"
}

for target in ["Pa", "Pd", "wt"]:
    y = df[target]

    rf_scores = cross_validate(rf, X, y, cv=cv, scoring=scoring)
    xgb_scores = cross_validate(xgb, X, y, cv=cv, scoring=scoring)

    print(f"\nTarget: {target}")

    print("Random Forest:")
    print("Mean R2:", round(rf_scores["test_r2"].mean(), 3))
    print("Mean MAE:", round(-rf_scores["test_neg_mae"].mean(), 3))
    print("Mean RMSE:", round(-rf_scores["test_neg_rmse"].mean(), 3))

    print("\nXGBoost:")
    print("Mean R2:", round(xgb_scores["test_r2"].mean(), 3))
    print("Mean MAE:", round(-xgb_scores["test_neg_mae"].mean(), 3))
    print("Mean RMSE:", round(-xgb_scores["test_neg_rmse"].mean(), 3))


Target: Pa
Random Forest:
Mean R2: 0.774
Mean MAE: 0.158
Mean RMSE: 0.28

XGBoost:
Mean R2: 0.89
Mean MAE: 0.115
Mean RMSE: 0.187

Target: Pd
Random Forest:
Mean R2: 0.828
Mean MAE: 0.09
Mean RMSE: 0.152

XGBoost:
Mean R2: 0.905
Mean MAE: 0.069
Mean RMSE: 0.114

Target: wt
Random Forest:
Mean R2: 0.67
Mean MAE: 0.117
Mean RMSE: 0.162

XGBoost:
Mean R2: 0.748
Mean MAE: 0.101
Mean RMSE: 0.143
